# B06-03 | 실제 OD600 성장곡선의 성장 특성 추정과 모델 적합 (v1)

**사례:** Aida–Ying *E. coli* BW25113, 7개 Round · 13,608개 곡선 · 1,029개 배지 조건. B06-02 Clean v2의 승인된 매핑·QC를 입력으로 사용한다.

**핵심 구별:** (i) 국소 로그 기울기에서 얻는 **탐색적 성장 지표**와 (ii) 선택된 곡선에 적합하는 **경험적 로지스틱 곡선 파라미터**는 원 논문의 `r`·`K`, 실제 최대 비성장률, 세대시간, 생물학적 lag time과 동일하다고 주장하지 않는다.

**학습 목표:** 결측 및 불규칙 관측에 안전한 추정, 적합 실패의 명시, 라운드/조건 반복 구조, 모델 진단과 비교 범위, CSV 저장 후 재검증.

**실행 환경:** conda `bioimage-ai`, Python 3.11; numpy, pandas, scipy, matplotlib, openpyxl. 원본 입력 9개 Excel과 B06-01 승인 매핑, B06-02 Clean v2 CSV가 로컬에 있어야 한다.

## 1. 실행 환경 및 선행 검증 확인

In [ ]:
from pathlib import Path
import re, json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.stats import pearsonr

cwd=Path.cwd().resolve()
ROOT=next((p for p in [cwd,cwd.parent,cwd/'bioimage-ai'] if (p/'data'/'bioculture'/'media_aida_ying_2025').is_dir()),None)
if ROOT is None: raise FileNotFoundError('프로젝트의 data/bioculture/media_aida_ying_2025를 찾을 수 없습니다.')
DATA=ROOT/'data'/'bioculture'/'media_aida_ying_2025'
PREV=ROOT/'results'/'bioculture'/'B06_02_growth_curves_v2'
AUDIT=ROOT/'results'/'bioculture'/'B06_01_link_audit_v2'
DEST=ROOT/'results'/'bioculture'/'B06_03_growth_models_v2'
DEST.mkdir(parents=True,exist_ok=True)
INPUT_CSV=PREV/'VALIDATED_curve_metrics_with_metadata.csv'
PREV_REPORT=PREV/'B06_02_validation_report.json'
APPROVAL=AUDIT/'link_audit_report_v2.json'
ROUND_FILES=sorted(DATA.glob('BW25113_Growth_Round*.xlsx'))
assert len(ROUND_FILES)==7 and all(p.exists() for p in [INPUT_CSV,PREV_REPORT,APPROVAL])
assert json.loads(APPROVAL.read_text(encoding='utf-8')).get('join_validated') is True
previous=json.loads(PREV_REPORT.read_text(encoding='utf-8'))
assert previous.get('qc_flags_persisted_and_verified') is True
print('ROOT:',ROOT,'\nOUTPUT:',DEST)
print('Previous report candidate count:',previous.get('model_candidate_curves_exploratory'))

## 2. 승인된 입력 CSV와 곡선 ID 정합성 확인

모델 후보는 B06-02의 교육용 기준을 따르며, 각 모델의 별도 진단 규칙도 사용한다. CSV의 ID 열은 **문자열로 보존**한다.

In [ ]:
metrics=pd.read_csv(INPUT_CSV,encoding='utf-8-sig',dtype={'Label':'string','Assay ID':'string','Condition ID':'string'},low_memory=False)
required=['round','Label','Assay ID','Condition ID','n_valid','flag_model_candidate','K','r']
assert all(c in metrics for c in required),[c for c in required if c not in metrics]
assert len(metrics)==13608 and not metrics.duplicated(['round','Label']).any()
def parse_bool(series):
    r=series.astype(str).str.strip().str.lower().map({'true':True,'false':False})
    if r.isna().any(): raise ValueError('불분명한 QC 플래그 값 발견')
    return r.astype(bool)
metrics['flag_model_candidate']=parse_bool(metrics['flag_model_candidate'])
assert metrics['Condition ID'].nunique()==1029
lookup=metrics.set_index(['round','Label'])
print('Total:',len(metrics),'Candidates:',metrics['flag_model_candidate'].sum())
display(metrics[['round','Label','Assay ID','Condition ID','n_valid','flag_model_candidate']].head())

## 3. 성장 특성의 운영상 정의

- `local_log_slope_max_per_h`: **양수 관측값의 연속된 5개 유효 시점**에 대해 `ln(OD)` 대 시간의 기울기를 계산하고 그중 최대값을 택한다. 이는 *탐색적 국소 로그 기울기*이지 검증된 생리학적 최대 비성장률은 아니다. 관측 구멍을 가로지르는 창은 허용하지 않는다.
- `growth_window_start_h`, `growth_window_end_h`: 위 최대 기울기 창의 관측 범위.
- `lag_proxy_h`: 로지스틱 모델의 중간점 접선이 초기 예측 기준값과 만나는 시간을 비음수로 제한한 **기하학적 지표**. 생물학적 적응 지연시간과 동일하지 않다.
- `model_K`, `model_rate_per_h`, `model_mid_h`: 관측 OD를 단순 3-파라미터 로지스틱 함수 `K/(1+exp[-rate*(t-mid)])`로 근사하는 파라미터. 초기 OD 오프셋을 포함하지 않아 생물학적 해석의 한계가 있다.

모델 파라미터는 실제 관측 범위에 대한 경험적 근사에 지나지 않는다. 성장률의 단위는 `h⁻¹`이며 B03 단일세포의 `min⁻¹`와 혼합하지 않는다.

## 4. 계산 함수와 합성 곡선 단위 테스트

In [ ]:
WINDOW_N=5
MIN_LOG_POINTS=5
# 로지스틱 적합은 계산량을 고려해 라운드별 조건 분산 표본에 수행(전체 곡선은 국소 기울기 계산).
# 실제 모델 적합 대상을 모두 지정하려면 MAX_FITS_PER_ROUND=None; 실행 시간이 크게 증가할 수 있다.
MAX_FITS_PER_ROUND=120
RANDOM_SEED=20261019
QUALITY_R2_MIN=0.80
QUALITY_NRMSE_MAX=0.15
MIN_MODEL_POINTS=12

def local_log_slope(t,y,n=WINDOW_N):
    t=np.asarray(t,float); y=np.asarray(y,float)
    assert len(t)==len(y)
    valid=np.isfinite(t)&np.isfinite(y)&(y>0)
    slopes=[]
    # 원본 인덱스에서 연속한 측정 시점만 사용; 누락점을 건너뛰지 않는다.
    for k in range(len(y)-n+1):
        sl=slice(k,k+n)
        if not valid[sl].all():continue
        tt=t[sl]; yy=np.log(y[sl]); duration=tt[-1]-tt[0]
        if duration<=0:continue
        slope=float(np.polyfit(tt,yy,1)[0]); slopes.append((slope,float(tt[0]),float(tt[-1])))
    if not slopes:return {'local_log_slope_max_per_h':np.nan,'growth_window_start_h':np.nan,'growth_window_end_h':np.nan,'n_valid_windows':0}
    best=max(slopes,key=lambda z:z[0])
    return {'local_log_slope_max_per_h':best[0], 'growth_window_start_h':best[1], 'growth_window_end_h':best[2],'n_valid_windows':len(slopes)}

def logistic(t,K,rate,mid):
    return K/(1.0+np.exp(np.clip(-rate*(t-mid),-700,700)))

def fit_logistic(t,y):
    t=np.asarray(t,float);y=np.asarray(y,float)
    ok=np.isfinite(t)&np.isfinite(y)
    t=t[ok];y=y[ok]
    bad={'fit_status':'not_fit','model_K':np.nan,'model_rate_per_h':np.nan,'model_mid_h':np.nan,'lag_proxy_h':np.nan,'lag_proxy_unclipped_h':np.nan,'rmse':np.nan,'nrmse_range':np.nan,'r2':np.nan,'n_fit':len(t),'bound_hit':False,'fit_quality_status':'not_evaluated'}
    if len(t)<MIN_MODEL_POINTS or t.max()-t.min()<6 or np.ptp(y)<0.02 or np.max(y)<=0:
        bad['fit_status']='insufficient_dynamic_range_or_points'; return bad
    K0=max(float(np.percentile(y,95)),0.01)
    upper=max(5*K0,2.0,float(np.max(y))*2)
    mid0=float(np.median(t))
    try:
        opt=least_squares(lambda p:logistic(t,*p)-y,x0=[K0,0.15,mid0],
          bounds=([0.000001,0.00001,float(t.min())-24],[upper,10,float(t.max())+24]),
          max_nfev=250,ftol=1e-7,xtol=1e-7,gtol=1e-7)
        if not opt.success:bad['fit_status']='optimizer_not_converged';return bad
        K,r,mid=map(float,opt.x);pred=logistic(t,K,r,mid)
        ss=float(np.sum((y-pred)**2));den=float(np.sum((y-np.mean(y))**2))
        r2=1-ss/den if den>0 else np.nan
        # tangent at the logistic midpoint: y(mid)=K/2; tangent slope=Kr/4
        # intercept with chosen baseline = first observed nonnegative OD, not zero OD.
        baseline=max(0.,float(y[0]));slope=K*r/4
        lag=float(mid+(baseline-K/2)/slope) if slope>0 else np.nan
        return {'fit_status':'fit_ok','model_K':K,'model_rate_per_h':r,'model_mid_h':mid,
          'lag_proxy_h':max(0.0,lag),'lag_proxy_unclipped_h':lag,
          'rmse':float(np.sqrt(np.mean((y-pred)**2))),
          'nrmse_range':float(np.sqrt(np.mean((y-pred)**2))/np.ptp(y)),
          'r2':r2,'n_fit':len(t),
          'bound_hit':bool(np.any(opt.active_mask!=0) or np.any(np.isclose(opt.x,[0.000001,0.00001,float(t.min())-24],rtol=0,atol=[1e-5,1e-4,0.02])) or np.any(np.isclose(opt.x,[upper,10,float(t.max())+24],rtol=0,atol=[max(upper*0.005,0.005),0.05,0.02]))),
          'fit_quality_status':'pending_quality_audit'}
    except (ValueError,FloatingPointError,OverflowError) as exc:
        bad['fit_status']='optimizer_exception_'+type(exc).__name__;return bad

# 테스트: 단조 증가, 결측 포함, 상수와 전체 결측 모두 조용히 다룸
xt=np.arange(0.5,20.5,0.5); yt=logistic(xt,0.8,0.5,10)
assert local_log_slope(xt,yt)['n_valid_windows']>0
assert fit_logistic(xt,yt)['fit_status']=='fit_ok'
assert local_log_slope(xt,np.full_like(xt,np.nan))['n_valid_windows']==0
assert fit_logistic(xt,np.full_like(xt,np.nan))['fit_status']!='fit_ok'
assert fit_logistic(xt,yt)['r2']>0.99
print('Function-level tests passed')

## 5. 적합 대상 추출 — 사전에 고정한 계층적 결정론적 표본

전체 13,608개에는 가벼운 국소 기울기를 적용한다. 로지스틱 최적화는 계산량과 교육 목적 때문에 **라운드당 최대 120개**의 탐색적 표본만 적합한다. 조건과 라운드별 표본 수 및 표본 추출 규칙을 CSV로 남긴다. 이 표본 결과를 13,608개 전체의 모델 성능으로 일반화하지 않는다.

In [ ]:
# 라운드별 고정 시드 무작위 선택: Condition ID 순서에 따른 편향을 피하되,
# 조건별 한 곡선을 먼저 뽑아 가능한 한 다양한 조건을 포함한다.
# 무작위 선택 역시 층화·생물학적 독립성·전체 대표성을 보장하지 않는다.
eligible=metrics.loc[metrics['flag_model_candidate']].copy()
sample_parts=[]
for rid,g in eligible.groupby('round',sort=True):
    rng=np.random.default_rng(RANDOM_SEED+int(rid))
    sg=g.sort_values(['Condition ID','Label']).copy()
    sg['_priority']=rng.random(len(sg))
    first=sg.sort_values('_priority').drop_duplicates('Condition ID',keep='first')
    first=first.sort_values('_priority')
    if MAX_FITS_PER_ROUND is not None:
        first=first.head(MAX_FITS_PER_ROUND)
        if len(first)<MAX_FITS_PER_ROUND:
            rest=sg.loc[~sg.index.isin(first.index)].sort_values('_priority').head(MAX_FITS_PER_ROUND-len(first))
            first=pd.concat([first,rest])
    sample_parts.append(first)
fit_sample=pd.concat(sample_parts,ignore_index=True)[['round','Label','Condition ID']]
assert not fit_sample.duplicated(['round','Label']).any()
fit_sample.to_csv(DEST/'model_fit_sample_keys.csv',index=False,encoding='utf-8-sig')
fit_keys=set(map(tuple,fit_sample[['round','Label']].itertuples(index=False,name=None)))
print('Random seed:',RANDOM_SEED,'Sample total:',len(fit_sample),'by round:',fit_sample.groupby('round').size().to_dict())

## 6. 7개 Excel 파일 분석 및 적합 결과 기록

각 파일을 한번씩 로딩하며, 승인된 원본 키가 모든 열에 존재하는지 재검사한다. 표본 외 곡선의 `fit_status=not_selected_for_fit`은 실패가 아니라 설계에 따른 미적합 상태다.

In [ ]:
rows=[]
for file in ROUND_FILES:
    rid=int(re.search(r'Round(\d+)',file.name,re.I).group(1))
    df=pd.read_excel(file,engine='openpyxl')
    tcol=next((c for c in df if str(c).strip().lower()=='time (h)'),None)
    assert tcol is not None
    t=pd.to_numeric(df[tcol],errors='coerce').to_numpy(float)
    assert np.isfinite(t).all() and np.all(np.diff(t)>0)
    labels=[c for c in df if re.fullmatch(r'Curve\d+',str(c),re.I)]
    expected=set(metrics.loc[metrics['round']==rid,'Label'].astype(str))
    assert set(map(str,labels))==expected
    vals=df[labels].apply(pd.to_numeric,errors='coerce')
    for label in labels:
        y=vals[label].to_numpy(float)
        base={'round':rid,'Label':str(label)}
        local=local_log_slope(t,y)
        if (rid,str(label)) in fit_keys:
            model=fit_logistic(t,y)
        else:
            model={'fit_status':'not_selected_for_fit','model_K':np.nan,'model_rate_per_h':np.nan,
               'model_mid_h':np.nan,'lag_proxy_h':np.nan,'lag_proxy_unclipped_h':np.nan,
               'rmse':np.nan,'nrmse_range':np.nan,'r2':np.nan,'n_fit':0,
               'bound_hit':False,'fit_quality_status':'not_evaluated'}
        rows.append({**base,**local,**model})
    print(f'Round {rid:02d}: {len(labels)} curves processed')
results=pd.DataFrame(rows)
assert len(results)==len(metrics) and not results.duplicated(['round','Label']).any()
merged=metrics.merge(results,on=['round','Label'],validate='one_to_one',how='left',indicator=True)
assert merged['_merge'].eq('both').all()
merged=merged.drop(columns='_merge')
merged['flag_local_growth_positive']=merged['local_log_slope_max_per_h']>0
# 적합 성공과 적합 품질을 서로 다르게 표시한다.
merged['flag_logistic_good_exploratory']=(merged['fit_status'].eq('fit_ok')
    & merged['r2'].ge(QUALITY_R2_MIN) & merged['nrmse_range'].le(QUALITY_NRMSE_MAX)
    & ~merged['bound_hit'].fillna(False).astype(bool))
merged['fit_quality_status']=np.select([
    ~merged['fit_status'].eq('fit_ok'),
    merged['bound_hit'].fillna(False).astype(bool),
    ~merged['r2'].ge(QUALITY_R2_MIN),
    ~merged['nrmse_range'].le(QUALITY_NRMSE_MAX),
],['not_fit','parameter_boundary','r2_below_threshold','nrmse_above_threshold'],default='passed_exploratory')
assert not (merged['flag_logistic_good_exploratory'] & ~merged['fit_status'].eq('fit_ok')).any()
results.to_csv(DEST/'curve_growth_model_features.csv',index=False,encoding='utf-8-sig')
merged.to_csv(DEST/'curve_growth_features_with_metadata.csv',index=False,encoding='utf-8-sig')
print('Local valid:',results['local_log_slope_max_per_h'].notna().sum(),
      'Fits:',results['fit_status'].value_counts().to_dict())
display(merged[['round','Label','Condition ID','local_log_slope_max_per_h','fit_status','model_rate_per_h','lag_proxy_h','r2']].head())

## 7. 모델 진단, 실패·선택 편향, K/r 탐색적 비교

### 7. 품질·편향·성장률 민감도 감사 (Clean v2)

- **적합 성공**은 최적화 종료에 불과하다. `R²`, 관측값 범위로 정규화한 RMSE, 경계 접근 여부를 동시에 점검한다.
- **지연시간 대용값**은 로지스틱 접선의 교점이다. `lag_proxy_unclipped_h`를 추가 보존하며 음수는 원래 식의 결과로 남긴다. `lag_proxy_h`의 0 절단 여부를 평가한다.
- **성장률 민감도**는 결측을 건너뛰지 않는 3·5·7개 연속 관측점의 최대 로그 기울기를 비교한다. 음수·영점은 로그에 포함하지 않는다.
- **표본 선정 편향**은 선정 및 비선정 집단의 라운드별 분포, 성장곡선 품질과 배지 조건 수를 비교한다. 확률 표집으로 바꾸었어도 생물학적 대표성을 증명하는 것은 아니다.

In [ ]:
# 창 크기 민감도: 각 라운드를 한 번씩만 읽으며 모델 적합과 독립적으로 산출
sensitivity=[]
for file in ROUND_FILES:
    rid=int(re.search(r'Round(\d+)',file.name,re.I).group(1))
    df=pd.read_excel(file,engine='openpyxl')
    t=pd.to_numeric(df['Time (h)'],errors='coerce').to_numpy(float)
    labels=[c for c in df if re.fullmatch(r'Curve\d+',str(c),re.I)]
    for label in labels:
        y=pd.to_numeric(df[label],errors='coerce').to_numpy(float)
        rec={'round':rid,'Label':str(label)}
        for wn in (3,5,7):
            rec[f'local_slope_w{wn}']=local_log_slope(t,y,n=wn)['local_log_slope_max_per_h']
        sensitivity.append(rec)
sens=pd.DataFrame(sensitivity)
assert len(sens)==len(metrics)
sens.to_csv(DEST/'local_slope_window_sensitivity.csv',index=False,encoding='utf-8-sig')
paired_sens=sens.dropna(subset=['local_slope_w3','local_slope_w5','local_slope_w7']).copy()
paired_sens['delta_w3_w7']=paired_sens['local_slope_w3']-paired_sens['local_slope_w7']
sens_summary={'n_all_three_windows':int(len(paired_sens)),
    'median_w3':float(paired_sens['local_slope_w3'].median()),
    'median_w5':float(paired_sens['local_slope_w5'].median()),
    'median_w7':float(paired_sens['local_slope_w7'].median()),
    'median_w3_minus_w7':float(paired_sens['delta_w3_w7'].median())}
(DEST/'local_slope_sensitivity_report.json').write_text(json.dumps(sens_summary,ensure_ascii=False,indent=2),encoding='utf-8')
print('Window sensitivity:',sens_summary)

# 데이터 수준의 모델 적합 품질 구분
quality=merged.loc[merged['fit_status'].eq('fit_ok'),
    ['round','Label','Condition ID','r2','rmse','nrmse_range','bound_hit',
     'lag_proxy_h','lag_proxy_unclipped_h','fit_quality_status','flag_logistic_good_exploratory']].copy()
quality['lag_was_clipped_to_zero']=quality['lag_proxy_unclipped_h']<0
quality.to_csv(DEST/'logistic_fit_quality_audit.csv',index=False,encoding='utf-8-sig')
print('Fit quality categories:',merged['fit_quality_status'].value_counts().to_dict())
print('Lag proxy clipped:',int(quality['lag_was_clipped_to_zero'].sum()))

# 모델 선정 편향 감사: 후보 곡선 내 선정/미선정 분포 (인과적 검증이 아님)
selection=metrics.copy()
selection['selected_for_fit']=[(int(r),str(l)) in fit_keys for r,l in
    selection[['round','Label']].itertuples(index=False,name=None)]
selection_audit=selection.groupby(['round','selected_for_fit'],as_index=False).agg(
    n_curves=('Label','size'),n_conditions=('Condition ID','nunique'),
    median_n_valid=('n_valid','median'),median_reference_K=('K','median'),
    median_reference_r=('r','median'))
selection_audit.to_csv(DEST/'model_selection_bias_audit.csv',index=False,encoding='utf-8-sig')
display(selection_audit,quality['fit_quality_status'].value_counts().to_frame('n'))

In [ ]:
fit_counts=merged.groupby(['round','fit_status']).size().rename('n').reset_index()
fit_counts.to_csv(DEST/'fit_status_by_round.csv',index=False,encoding='utf-8-sig')
paired=[]
for source,estimated in [('K','model_K'),('r','model_rate_per_h'),('r','local_log_slope_max_per_h')]:
    a=pd.to_numeric(merged[source],errors='coerce');b=pd.to_numeric(merged[estimated],errors='coerce')
    good=np.isfinite(a)&np.isfinite(b)
    corr=float(np.corrcoef(a[good],b[good])[0,1]) if good.sum()>=3 and np.std(a[good])>0 and np.std(b[good])>0 else np.nan
    paired.append({'reference':source,'estimated':estimated,'n_pairs':int(good.sum()),'pearson_exploratory':corr,
                   'reference_definition_confirmed':False})
pair_stats=pd.DataFrame(paired)
pair_stats.to_csv(DEST/'reference_metric_model_comparison.csv',index=False,encoding='utf-8-sig')
display(fit_counts,pair_stats)
# 참고: 같은 Condition ID에서 나온 곡선을 독립 생물학적 반복으로 취급하지 않는다.
cond=merged.groupby('Condition ID',as_index=False).agg(n_curves=('Label','size'),n_rounds=('round','nunique'),
   n_local_estimates=('local_log_slope_max_per_h','count'),median_local_log_slope=('local_log_slope_max_per_h','median'),
   n_model_fit_ok=('fit_status',lambda s:int((s=='fit_ok').sum())),median_logistic_rate=('model_rate_per_h','median'))
cond.to_csv(DEST/'condition_exploratory_growth_summary.csv',index=False,encoding='utf-8-sig')
print('Conditions:',len(cond))

## 8. 실제 곡선과 선택적 로지스틱 적합 시각화

In [ ]:
ok=merged.loc[merged['fit_status'].eq('fit_ok')].sort_values(['round','Label']).groupby('round',as_index=False).head(1)
fig,axes=plt.subplots(1,max(1,len(ok)),figsize=(4*max(1,len(ok)),3.4),squeeze=False)
for ax,(_,rec) in zip(axes[0],ok.iterrows()):
    rid=int(rec['round']);file=next(p for p in ROUND_FILES if int(re.search(r'Round(\d+)',p.name,re.I).group(1))==rid)
    src=pd.read_excel(file,engine='openpyxl',usecols=['Time (h)',str(rec['Label'])])
    t=src['Time (h)'].to_numpy(float);y=pd.to_numeric(src[str(rec['Label'])],errors='coerce').to_numpy(float)
    valid=np.isfinite(y)
    ax.plot(t[valid],y[valid],'.',ms=3,label='Observed')
    xx=np.linspace(t.min(),t.max(),200);ax.plot(xx,logistic(xx,rec['model_K'],rec['model_rate_per_h'],rec['model_mid_h']),lw=1.3,label='Logistic')
    ax.set_title(f'Round {rid} / {rec["Label"]}');ax.set_xlabel('Time (h)');ax.set_ylabel('Observed OD')
    ax.legend(fontsize=8)
if not len(ok): axes[0,0].text(0.1,0.5,'No successful fits')
fig.tight_layout();fig.savefig(DEST/'logistic_fit_examples.png',dpi=160,bbox_inches='tight');plt.show()
fig,ax=plt.subplots(figsize=(7,4))
data=[merged.loc[(merged['round']==rid)&np.isfinite(merged['local_log_slope_max_per_h']),'local_log_slope_max_per_h'].to_numpy() for rid in sorted(merged['round'].unique())]
ax.boxplot(data,showfliers=False);ax.set_xticklabels(sorted(merged['round'].unique()));ax.set_xlabel('Round');ax.set_ylabel('Maximum local log slope (1/h)')
ax.set_title('Exploratory local slopes, by round (not physiological max rate)')
fig.tight_layout();fig.savefig(DEST/'local_growth_rate_by_round.png',dpi=160,bbox_inches='tight');plt.show()

## 9. 저장물 재조회 및 입력 파일 SHA256 감사

In [ ]:
disk=pd.read_csv(DEST/'curve_growth_features_with_metadata.csv',encoding='utf-8-sig',dtype={'Label':'string','Assay ID':'string','Condition ID':'string'})
assert len(disk)==13608 and not disk.duplicated(['round','Label']).any()
assert disk['Condition ID'].nunique()==1029
assert disk['fit_status'].notna().all()
assert disk['fit_status'].eq('fit_ok').sum()==merged['fit_status'].eq('fit_ok').sum()
assert len(fit_sample)==sum(disk['fit_status']!='not_selected_for_fit')
assert len(pd.read_csv(DEST/'logistic_fit_quality_audit.csv'))==int(disk['fit_status'].eq('fit_ok').sum())
assert len(pd.read_csv(DEST/'local_slope_window_sensitivity.csv'))==len(disk)
assert disk.loc[~disk['flag_model_candidate'].astype(str).str.lower().eq('true'),'fit_status'].eq('not_selected_for_fit').all()
summary={'dataset':'Aida–Ying BW25113','n_curves':len(disk),'n_conditions':int(disk['Condition ID'].nunique()),
 'n_fit_sample':len(fit_sample),'n_fit_ok':int(disk['fit_status'].eq('fit_ok').sum()),
 'n_local_estimates':int(disk['local_log_slope_max_per_h'].notna().sum()),
 'n_quality_passed':int(disk['flag_logistic_good_exploratory'].astype(str).str.lower().eq('true').sum()),
 'n_boundary_hits':int(quality['bound_hit'].sum()),
 'n_lag_clipped_to_zero':int(quality['lag_was_clipped_to_zero'].sum()),
 'window_sensitivity':sens_summary,
 'fit_sampling':'seeded pseudorandom selection, one per condition first, max 120 per round, not independently representative',
 'scientific_metric_definition_confirmed':False,
 'interpretation':'Local slope and logistic parameters are exploratory computational metrics; not validated biological maximum rate, lag time, or original K/r.'}
(DEST/'B06_03_validation_report.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
input_paths=ROUND_FILES+[INPUT_CSV,PREV_REPORT,APPROVAL]
pd.DataFrame([{'file':str(p.relative_to(ROOT)),'sha256':sha256(p),'bytes':p.stat().st_size} for p in input_paths]).to_csv(DEST/'input_manifest_sha256.csv',index=False,encoding='utf-8-sig')
pd.DataFrame([{'file':p.name,'sha256':sha256(p),'bytes':p.stat().st_size} for p in sorted(DEST.iterdir()) if p.is_file() and p.name!='output_manifest_sha256.csv']).to_csv(DEST/'output_manifest_sha256.csv',index=False,encoding='utf-8-sig')
print('Disk re-read checks passed.\n',json.dumps(summary,ensure_ascii=False,indent=2))

## 10. 결과 해석, 제약 및 다음 단계

1. 로지스틱 모델은 **사전 정의한 하위 표본**에만 적합하였다. 표본 밖 곡선의 미적합은 오류가 아니다.
2. 세 파라미터 로지스틱 함수는 초기 배경 OD, 감소기 또는 복잡한 이중 성장(diauxic growth)을 충분히 표현하지 못할 수 있다. 높은 R²도 생물학적 메커니즘의 검증이 아니다.
3. 창 길이, 양수 OD 사용, 연속 시점 요구사항, QC 후보 기준은 **분석자가 정한 탐색적 정의**이다. 실제 성장률 추정은 배경 보정, 고정된 모델 비교, 독립 반복 설계 검토를 거쳐야 한다.
4. 원 데이터의 `K`, `r`, `K_info`, `r_info` 정의 확인은 별도 문헌·메타데이터 작업으로 남겨둔다.
5. 모든 배지 조성에 대한 예측, 인과적 배지 효과, 최적 배지 검증은 아직 수행하지 않았다.

### 연습문제 — 정답 및 예제 코드는 추후 B06-03 Word 해설서에 포함
1. `WINDOW_N=5`를 3과 7로 바꾸면 국소 로그 기울기의 분포가 어떻게 바뀌는가? 특히 노이즈에 주의하라.
2. `fit_status`별 수를 집계하고 빈 곡선·샘플 제외·적합 실패를 구분하라.
3. Round 간 최대 국소 로그 기울기 중앙값 차이가 배지의 인과적 효과를 증명하지 못하는 이유를 설명하라.
4. 로지스틱 적합의 R²가 높은 곡선과 낮은 곡선을 각각 골라 원본 점을 시각화하라.
5. `Condition ID`별 반복 수가 편중될 경우, 곡선별 평균과 조건별 평균이 어떻게 달라지는지 설명하라.